### Steps to run this on NIRD:

1. Create a user account at https://www.metacenter.no/user/application/form/storage/, having previously communicated with a person that has a project there and can accept your request.

2. Open a terminal window and ssh conect to @login.nird.sigma2.no

3. Type 'module load Miniconda3/23.10.0-1'

4. Type 'source activate' (onda activate/deactivate will not work unless you do 3, 4)

5. Type 'conda activate your_environment'

6. Type 'jupyter lab --port=8080' (jupyter lab has to be installed in the environment first)

7. Open a new terminal window (so you're on local) and start listening to the port on nird: 'ssh -L 8080:localhost:8080 username@loginX.nird.sigma2.no',  X is the number of the login node you run jupyter on.

8. Open the link from the nird window in browser or just do 'localhost:8080' in your browser.

### **Step 0.1: Intalling necessary packages into the new NIRD environment**

In [1]:
#!pip install cfgrib
#!pip install cftime
#!pip install xesmf
#!pip install scikit-learn 
#!pip install dask
#!pip install matplotlib
#!conda clean --all -y

#!conda install -c conda-forge esmpy xesmf -y
#!conda install -c conda-forge esmpy

### **Step 0.2: Importing packages**

In [1]:
# for navigating into directories
import os
from pathlib import Path
#for dealing with .nc data
import cfgrib
import cftime
import xarray as xr
#for dealing with panda arrays
import numpy as np
import pandas as pd
#for math
import xesmf as xe                             # for regridding
from glob import glob
from scipy import integrate
from scipy.stats import linregress
from scipy.stats import pearsonr
from scipy.interpolate import interp1d
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import dask
#for ignoring stuff
import warnings
warnings.simplefilter("ignore")
# for skiping running a certain cell
from IPython.core.magic import register_cell_magic

@register_cell_magic
def comment(line, cell):
    return
    
# for saving lists
import pickle 

/nird/home/chrikap/.conda/envs/nird_heat_transport/lib/python3.12/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


### **Step 1.1: Accessing the data from NIRD**

In [6]:
ls '/datalake/NS9560K/ESGF/CMIP6/CMIP'

AS-RCEC/  CCCR-IITM/     E3SM-Project/         KIOST/      NCAR/       THU/
AWI/      CMCC/          EC-Earth-Consortium/  MIROC/      NCC/        UA/
BCC/      cmip-data/     FIO-QLNM/             MOHC/       NIMS-KMA/
CAMS/     CNRM-CERFACS/  HAMMOZ-Consortium/    MPI-M/      NOAA-GFDL/
CAS/      CSIRO/         INM/                  MRI/        NUIST/
CCCma/    CSIRO-ARCCSS/  IPSL/                 NASA-GISS/  SNU/


In [2]:
# Path of the CMIP6 data
cmip_path = Path('/datalake/NS9560K/ESGF/CMIP6/CMIP')

# Get all team names, excluding 'cmip-data'
groups = [group for group in sorted(os.listdir(cmip_path)) if group != 'cmip-data']

# Unzipping the models in each group
models_all = [sorted(model for model in sorted(os.listdir(f'{cmip_path}/{group}')) if model !='README') for group in groups]

models_allOpen a new terminal window (so you're on local) and start listening to the port on nird: 'ssh -L 8080:localhost:8080 username@loginX.nird.sigma2.no', X is the number of the login node you run jupyter on.

Open the link from the nird window in browser or just do 'local

[['TaiESM1'],
 ['AWI-CM-1-1-MR', 'AWI-ESM-1-1-LR'],
 ['BCC-CSM2-MR', 'BCC-ESM1'],
 ['CAMS-CSM1-0'],
 ['CAS-ESM2-0', 'FGOALS-f3-L', 'FGOALS-g3'],
 ['IITM-ESM'],
 ['CanESM5', 'CanESM5-1', 'CanESM5-CanOE'],
 ['CMCC-CM2-HR4', 'CMCC-CM2-SR5', 'CMCC-ESM2'],
 ['CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1'],
 ['ACCESS-ESM1-5'],
 ['ACCESS-CM2'],
 ['E3SM-1-0', 'E3SM-1-1', 'E3SM-1-1-ECA'],
 ['EC-Earth3',
  'EC-Earth3-AerChem',
  'EC-Earth3-CC',
  'EC-Earth3-LR',
  'EC-Earth3-Veg',
  'EC-Earth3-Veg-LR'],
 ['FIO-ESM-2-0'],
 ['MPI-ESM-1-2-HAM'],
 ['INM-CM4-8', 'INM-CM5-0'],
 ['IPSL-CM5A2-INCA', 'IPSL-CM6A-LR', 'IPSL-CM6A-LR-INCA'],
 ['KIOST-ESM'],
 ['MIROC-ES2H', 'MIROC-ES2L', 'MIROC6'],
 ['HadGEM3-GC31-LL', 'HadGEM3-GC31-MM', 'UKESM1-0-LL', 'UKESM1-1-LL'],
 ['ICON-ESM-LR', 'MPI-ESM1-2-HR', 'MPI-ESM1-2-LR'],
 ['MRI-ESM2-0'],
 ['GISS-E2-1-G',
  'GISS-E2-1-G-CC',
  'GISS-E2-1-H',
  'GISS-E2-2-G',
  'GISS-E2-2-H',
  'GISS-E3-G'],
 ['CESM2', 'CESM2-FV2', 'CESM2-WACCM', 'CESM2-WACCM-FV2'],
 ['NorCPM1', 'N

### **Step 1.2: Finding which models have both experiments, both realms, and all of the variables for the same ensemble member (r1i1p1f1)**

In [4]:
experiments = ['piControl', 'abrupt-4xCO2']
realms = ['Amon', 'Omon']
atm_vars = ['rsdt', 'rsut', 'rlut', 'hfls', 'hfss', 'rlds', 'rlus', 'rsus', 'rsds']
oc_vars = ['hfbasin']

# Catching the CMIP6 data, all except NCC:

cmip_models = []

for i in range (len(groups)): #go through all the teams
        
    for j in range(len(models_all[i])): #go through all the models
        
        model = models_all[i][j]
        model_path = f'{cmip_path}/{groups[i]}/{models_all[i][j]}'        

        if all(exp in os.listdir(model_path) for exp in experiments): #if they have both experiments
    
            experiment_path_1 = f'{model_path}/piControl'
            experiment_path_2 = f'{model_path}/abrupt-4xCO2'
        
            if 'r1i1p1f1' in os.listdir(experiment_path_1) and 'r1i1p1f1' in os.listdir(experiment_path_2): #if they have the ensemble we need
    
                ensemble_path_1 = f'{experiment_path_1}/r1i1p1f1'
                ensemble_path_2 = f'{experiment_path_2}/r1i1p1f1'
        
                if all(realm in os.listdir(ensemble_path_1) for realm in realms) and all(realm in os.listdir(ensemble_path_2) for realm in realms): #if they have
                                                                                                                                                    #all realms
                    atm_path_1 = f'{ensemble_path_1}/Amon'
                    atm_path_2 = f'{ensemble_path_2}/Amon'
                    oc_path_1 = f'{ensemble_path_1}/Omon'
                    oc_path_2 = f'{ensemble_path_2}/Omon'
                                    
                    if 'hfbasin' in os.listdir(oc_path_1) and all(atm_var in os.listdir(atm_path_1)for atm_var in atm_vars) and 'hfbasin' in os.listdir(oc_path_2) and all(atm_var in os.listdir(atm_path_2)for atm_var in atm_vars):
                        print(model, 'has everything') #if they have all variables
    
                        cmip_models.append(model)

# Catching the NorESM data:

ncc_path = '/projects/NS9034K/CMIP6/CMIP'
ncc_models = []

for model in ['NorCPM1', 'NorESM2-LM', 'NorESM2-MM']:

    ncc_ensemble_path_1 = f'{ncc_path}/NCC/{model}/piControl/r1i1p1f1'
    ncc_ensemble_path_2 = f'{ncc_path}/NCC/{model}/abrupt-4xCO2/r1i1p1f1'
    
    if all(realm in os.listdir(ncc_ensemble_path_1) for realm in realms) and all(realm in os.listdir(ncc_ensemble_path_2) for realm in realms): #if they have all realms
                        
        ncc_atm_path_1 = f'{ncc_ensemble_path_1}/Amon'
        ncc_atm_path_2 = f'{ncc_ensemble_path_2}/Amon'
        ncc_oc_path_1 = f'{ncc_ensemble_path_1}/Omon'
        ncc_oc_path_2 = f'{ncc_ensemble_path_2}/Omon'
                                    
        if 'hfbasin' in os.listdir(ncc_oc_path_1) and all(atm_var in os.listdir(ncc_atm_path_1)for atm_var in atm_vars) and 'hfbasin' in os.listdir(ncc_oc_path_2) and all(atm_var in os.listdir(ncc_atm_path_2)for atm_var in atm_vars) :
            print(model, 'has everything') #if they have all variables
    
            ncc_models.append(model)

FGOALS-g3 has everything
CanESM5 has everything
EC-Earth3-Veg has everything
IPSL-CM6A-LR has everything
MPI-ESM1-2-HR has everything
MRI-ESM2-0 has everything
SAM0-UNICON has everything
NorCPM1 has everything
NorESM2-LM has everything
NorESM2-MM has everything


### **Step 2: Getting the heat fluxes, zonally and time-averaged**

In [5]:
# I re-grid everything to a 2x2 grid, following this tutorial: https://xesmf.readthedocs.io/en/stable/notebooks/Rectilinear_grid.html

new_grid_amon = xr.Dataset({
        "lat": (["lat"], np.arange(-90, 92, 2), {"units": "degrees_north"}),
        "lon": (["lon"], np.arange(0, 362, 2), {"units": "degrees_east"})
    })

In [6]:
# Choosing the time period

# years 0-150:

start_y = 0
end_y = 1800

period = 1789

start_number = 0 # for plot labeling
end_number = 150

# years 50-150:

#start_y = 600
#end_y = 1800

#period = 1189 #(1200-12+1)

#start_number = 50 # for plot labeling
#end_number = 150

# years 100-150

#start_y = 1200
#end_y = 1800

#period = 589 #(600-12+1)

#start_number = 100 # for plot labeling
#end_number = 150

In [11]:
component_lists_exp = {comp: [] for comp in ['rlut', 'rsdt', 'rsut', 'hfls', 'hfss', 'rlds', 'rlus', 'rsds', 'rsus']}
lat_exp = []

for experiment in experiments:
    component_lists_model = {comp: [] for comp in component_lists_exp.keys()}
    lat_model = []

    for i in range (len(groups)): 
        group = groups[i]
            
        for j in range(len(models_all[i])):           
            model = models_all[i][j]

            if model in ncc_models: #different directory for the ncc (noresm) data
                model_path = f'{ncc_path}/{group}/{model}'
            elif model in cmip_models:
                model_path = f'{cmip_path}/{group}/{model}'  

            else:
                continue
    
            print(model)

            if model in ['EC-Earth3-Veg','NorCPM1','FGOALS-g3','SAM0-UNICON']: #models I ended up excluding for different reasons each, although they seemed qualified before
                continue

            components = atm_vars
            
            for component in components:
                print(component)    
        
                folder_path_0 = f'{model_path}/{experiment}/r1i1p1f1/Amon/{component}'                   
                        
                file_names = sorted(os.listdir(folder_path_0))

                exception_path = sorted(os.listdir(f'{folder_path_0}/{file_names[0]}'))

                if 'latest' not in exception_path: #if they don't have the 'latest' data, we just take whichever version is available

                    grid_folder_path = f'{folder_path_0}/{file_names[0]}/{exception_path[0]}'

                else:

                    grid_folder_path = f'{folder_path_0}/{file_names[0]}/latest'

                grid_file_names = sorted(os.listdir(grid_folder_path))
        
                # Using dask to read datasets
                datasets = [xr.open_dataset(os.path.join(grid_folder_path, file_name), chunks={'time': 10}) for file_name in grid_file_names if file_name.endswith('.nc')]
        
                if len(datasets) == 1:
                    ds = datasets[0]
                    ds_50 = ds.isel(time=slice(start_y, end_y))
                else:
                    # Merge all the datasets in the list along the same dimension
                    merged_datasets = xr.concat(datasets, dim='time')
                    ds_50 = merged_datasets.isel(time=slice(start_y, end_y))        

                # Perform regridding
                regridder = xe.Regridder(ds_50, new_grid_amon, "bilinear")
                component_regridded = regridder(ds_50, keep_attrs=True)

                # Use Dask to compute and save the file
                dask.compute()
        
                lon = component_regridded['lon'].values
                            
                lat = component_regridded['lat'].values
                            
                component_values = component_regridded[component].values
                            
                zonal_ave = (np.trapz(component_values, x=np.deg2rad(lon), dx=np.deg2rad(2), axis=2))/(2*np.pi) 
            
                #time_ave = zonal_ave.mean(axis=0)
                
                if component in component_lists_model:
                    #component_lists_model[component].append(time_ave)
                    component_lists_model[component].append(zonal_ave)
            
            lat_model.append(lat)
    
    for comp in component_lists_exp:
        component_lists_exp[comp].append(component_lists_model[comp])
    lat_exp.append(lat_model)

FGOALS-g3
CanESM5
rsdt
rsut
rlut
hfls
hfss
rlds
rlus
rsus
rsds
EC-Earth3-Veg
IPSL-CM6A-LR
rsdt
rsut
rlut
hfls
hfss
rlds
rlus
rsus
rsds
MPI-ESM1-2-HR
rsdt
rsut
rlut
hfls
hfss
rlds
rlus
rsus
rsds
MRI-ESM2-0
rsdt
rsut
rlut
hfls
hfss
rlds
rlus
rsus
rsds
NorCPM1
NorESM2-LM
rsdt
rsut
rlut
hfls
hfss
rlds
rlus
rsus
rsds
NorESM2-MM
rsdt
rsut
rlut
hfls
hfss
rlds
rlus
rsus
rsds
SAM0-UNICON
FGOALS-g3
CanESM5
rsdt
rsut
rlut
hfls
hfss
rlds
rlus
rsus
rsds
EC-Earth3-Veg
IPSL-CM6A-LR
rsdt
rsut
rlut
hfls
hfss
rlds
rlus
rsus
rsds
MPI-ESM1-2-HR
rsdt
rsut
rlut
hfls
hfss
rlds
rlus
rsus
rsds
MRI-ESM2-0
rsdt
rsut
rlut
hfls
hfss
rlds
rlus
rsus
rsds
NorCPM1
NorESM2-LM
rsdt
rsut
rlut
hfls
hfss
rlds
rlus
rsus
rsds
NorESM2-MM
rsdt
rsut
rlut
hfls
hfss
rlds
rlus
rsus
rsds
SAM0-UNICON


### Heat fluxes, time averaged

In [8]:
rsdts = np.array(component_lists_exp['rsdt'])
rsuts = np.array(component_lists_exp['rsut'])
rluts = np.array(component_lists_exp['rlut'])
hflss = np.array(component_lists_exp['hfls'])
hfsss = np.array(component_lists_exp['hfss'])
rldss = np.array(component_lists_exp['rlds'])
rluss = np.array(component_lists_exp['rlus'])
rsuss = np.array(component_lists_exp['rsus'])
rsdss = np.array(component_lists_exp['rsds'])

In [9]:
#Saving everything to my NIRD account. From there, I can also download them to local.

os.chdir('6_models')
np.save('rsdts', rsdts)
np.save('rsuts', rsuts)
np.save('rluts', rluts)
np.save('hflss', hflss)
np.save('hfsss', hfsss)
np.save('rldss', rldss)
np.save('rluss', rluss)
np.save('rsuss', rsuss)
np.save('rsdss', rsdss)

### Heat fluxes, time-dependent

You basically run the same script as above but comment out this line:

component_lists_model[component].append(time_ave)

and use this line, instead:#component_lists_model[component].append(zonal_ave)

can also comment out: time_ave = zonal_ave.mean(axis=0) to save some space and time

In [12]:
rsdts_t = np.array(component_lists_exp['rsdt'])
rsuts_t = np.array(component_lists_exp['rsut'])
rluts_t = np.array(component_lists_exp['rlut'])
hflss_t = np.array(component_lists_exp['hfls'])
hfsss_t = np.array(component_lists_exp['hfss'])
rldss_t = np.array(component_lists_exp['rlds'])
rluss_t = np.array(component_lists_exp['rlus'])
rsuss_t = np.array(component_lists_exp['rsus'])
rsdss_t = np.array(component_lists_exp['rsds'])

In [15]:
#Saving everything to my NIRD account. From there, I can also download them to local.
os.chdir('/nird/home/chrikap/6_models')
np.save('rsdts_t', rsdts_t)
np.save('rsuts_t', rsuts_t)
np.save('rluts_t', rluts_t)
np.save('hflss_t', hflss_t)
np.save('hfsss_t', hfsss_t)
np.save('rldss_t', rldss_t)
np.save('rluss_t', rluss_t)
np.save('rsuss_t', rsuss_t)
np.save('rsdss_t', rsdss_t)

### *We have to calculate hfbasin separately, because of the basins and the fact that the grid is different!!

In [16]:
# Renaming the global basins so they are all called the same
name_mapping = {
    "global_ocean": "global_ocean",
    "global": "global_ocean",
    "g": "global_ocean",
    "Global": "global_ocean"
}

# Function to process data and extract the global_ocean basin 
def basin_separation_I(data, model):
    global_data = []

    # Extract basin names
    basin_labels = ['atlantic_arctic_ocean', 'atlantic_arctic_extended_ocean', 'indian_pacific_ocean', 'global_ocean']
    basin_indexes = range(len(basin_labels))  # Create a default index range
    
    if model == 'IPSL-CM6A-LR':
        data = data.rename({'3basin': 'basin'})
        basin_labels = ['Global', 'Atlantic', 'Indo-Pacific']
        basin_indexes = range(len(basin_labels))  # Create a default index range    

    elif model == 'NorESM2-LM':
        basin_labels = ['atlantic_arctic_ocean', 'atlantic_arctic_extended_ocean', 'indian_pacific_ocean', 'global_ocean']
        basin_indexes = range(len(basin_labels))  # Create a default index range    
        
    elif 'sector' in data.coords and 'basin' in data.dims:
        basin_indexes = data['basin'].values
        basin_labels = data['sector'].values
        basin_labels = [label.decode('utf-8').strip() for label in basin_labels]
    elif 'basin' in data.dims and 'requested' in data['basin'].attrs:
        basin_indexes = data['basin'].values
        requested_attr = data['basin'].attrs['requested']
        basin_labels = [k.split('=')[0].strip() for k in requested_attr.split(', ')]
    else:
        print(f"{model} : No valid basin names found.")
        return []

    # Create a mapping of indexes to standardized names
    basin_mapping = {}
    
    for idx, label in zip(basin_indexes, basin_labels):
        standardized_name = name_mapping.get(label.lower(), f"unknown_{label}")
        basin_mapping[idx] = standardized_name

    # Extract data for the global_ocean basin
    for idx, name in basin_mapping.items():

        basin_data = data.isel(basin=idx)  # Extract data for the basin
        if name == "global_ocean":
            global_data.append(basin_data)

    return global_data

In [17]:
target_lat = np.arange(-90, 92, 2)  # Define the target latitude grid

In [21]:
#hfbasins_exp = [] # time average
hfbasins_exp_t = [] # no time average
lat_basin_exp = []

for experiment in experiments:
    hfbasins_model = []
    lat_basin_model = []

    for i, group in enumerate(groups):
        for j, model in enumerate(models_all[i]):
            
            if model in ncc_models:
                model_path = f'{ncc_path}/{group}/{model}'
            elif model in cmip_models:
                model_path = f'{cmip_path}/{group}/{model}'
            else:
                continue

            print(model)

            if model in ['EC-Earth3-Veg','NorCPM1','FGOALS-g3','SAM0-UNICON']: # models I ended up excluding for different reasons each, although they seemed qualified before
                continue
            
            folder_path_0 = f'{model_path}/{experiment}/r1i1p1f1/Omon/hfbasin'
            file_names = sorted(os.listdir(folder_path_0))
            exception_path = sorted(os.listdir(f'{folder_path_0}/{file_names[0]}'))
            
            if 'latest' not in exception_path:
                grid_folder_path = f'{folder_path_0}/{file_names[0]}/{exception_path[0]}'
            else:
                grid_folder_path = f'{folder_path_0}/{file_names[0]}/latest'
            
            grid_file_names = sorted(os.listdir(grid_folder_path))
            
            datasets = [xr.open_dataset(os.path.join(grid_folder_path, file_name), chunks={'time': 10}) 
                        for file_name in grid_file_names if file_name.endswith('.nc')]
            
            if len(datasets) == 1:
                ds = datasets[0]
            else:
                ds = xr.concat(datasets, dim='time')
            
            #ds_50 = ds.isel(time=slice(start_y, end_y)).mean(dim='time') #time average
            ds_50 = ds.isel(time=slice(start_y, end_y)) # no time average

            lat = ds_50['nav_lat'][:, 0].values if model == 'IPSL-CM6A-LR' else ds_50['lat'].values
            dask.compute()
            
            component_values = ds_50['hfbasin']
            #component_values = xr.where(np.isfinite(component_values), component_values, 0) #replacing nans and infs with 0, bc there's no ht happening on land
            global_data = basin_separation_I(component_values, model)
            
            for global_basin_data in global_data:

                if global_basin_data.shape == (332, 1):  #just for the ipsl model- i have to fix how t make it more general later
                    global_basin_data = global_basin_data[:, 0]

                hfbasins_model.append(global_basin_data.values * 10**-15) # has to happen now because interpolation will be difficult with such high values later
                lat_basin_model.append(lat)
    
    #hfbasins_exp.append(hfbasins_model) # time averaged
    hfbasins_exp_t.append(hfbasins_model) # time-dependent
    lat_basin_exp.append(lat_basin_model)

FGOALS-g3
CanESM5
EC-Earth3-Veg
IPSL-CM6A-LR
MPI-ESM1-2-HR
MRI-ESM2-0
NorCPM1
NorESM2-LM
NorESM2-MM
SAM0-UNICON
FGOALS-g3
CanESM5
EC-Earth3-Veg
IPSL-CM6A-LR
MPI-ESM1-2-HR
MRI-ESM2-0
NorCPM1
NorESM2-LM
NorESM2-MM
SAM0-UNICON


In [19]:
# Using time average
hfbasins_exp_interp = []  # Interpolated results storage

for hfbasins_model, lat_basin_model in zip(hfbasins_exp, lat_basin_exp):
    hfbasins_model_interp = []
    
    for hfbasin_data, lat_data in zip(hfbasins_model, lat_basin_model):

        # Convert lat_data to a pandas Series to drop duplicates while keeping indices
        lat_series = pd.Series(lat_data)
        unique_lat_data, unique_indices = np.unique(lat_series, return_index=True)

        # Sort by latitude to ensure proper interpolation
        sorted_indices = np.argsort(unique_lat_data)
        unique_lat_data = unique_lat_data[sorted_indices]
        hfbasin_data = hfbasin_data[unique_indices][sorted_indices]  # Align hfbasin_data

        # Now create DataArray and interpolate
        da = xr.DataArray(hfbasin_data, coords=[unique_lat_data], dims=["lat"])
        da_interp = da.interp(lat=target_lat)
        hfbasins_model_interp.append(da_interp.values)

    hfbasins_exp_interp.append(hfbasins_model_interp)

hfbasins_exp_interp = np.array(hfbasins_exp_interp)

In [20]:
os.chdir('/nird/home/chrikap/6_models')
np.save('hfbasins_exp_interp', hfbasins_exp_interp)

In [ ]:
#  Run again with no time average

In [22]:
# Using time average
hfbasins_exp_t_interp = []  # Interpolated results storage

for hfbasins_model, lat_basin_model in zip(hfbasins_exp_t, lat_basin_exp):
    hfbasins_model_interp = []
    
    for hfbasin_data, lat_data in zip(hfbasins_model, lat_basin_model):
        # Convert lat_data to a pandas Series to drop duplicates while keeping indices
        lat_series = pd.Series(lat_data)
        unique_lat_data, unique_indices = np.unique(lat_series, return_index=True)
        
        # Sort by latitude to ensure proper interpolation
        sorted_indices = np.argsort(unique_lat_data)
        unique_lat_data = unique_lat_data[sorted_indices]
        
        # Convert hfbasin_data to a NumPy array for indexing
        hfbasin_data = np.array(hfbasin_data)
        hfbasins_time_interp = []
        
        for hfbasin_time in hfbasin_data:
            hfbasin_time = np.array(hfbasin_time)[unique_indices][sorted_indices]  # Align hfbasin_data properly
            
            # Flatten if extra dimensions exist
            if hfbasin_time.ndim > 1:
                hfbasin_time = hfbasin_time.squeeze()
            
            # Ensure it aligns with unique_lat_data
            if hfbasin_time.shape[0] != unique_lat_data.shape[0]:
                raise ValueError(f"Mismatch in dimensions: hfbasin_time={hfbasin_time.shape}, unique_lat_data={unique_lat_data.shape}")
            
            # Ensure the correct shape for DataArray
            da = xr.DataArray(hfbasin_time, coords={"lat": unique_lat_data}, dims=["lat"])
            da_interp = da.interp(lat=target_lat)
            
            hfbasins_time_interp.append(da_interp.values)
        
        hfbasins_model_interp.append(hfbasins_time_interp)
    
    hfbasins_exp_t_interp.append(hfbasins_model_interp)

hfbasins_exp_t_interp = np.array(hfbasins_exp_t_interp)

In [23]:
# Save interpolated data
os.chdir('/nird/home/chrikap/6_models')
np.save('hfbasins_exp_t_interp', hfbasins_exp_t_interp)

In [ ]:
# Doing the same but for the atlantic arctic ocean...I know it's the same script but I want things simple for now...

In [24]:
# Renaming the atlantic arctic basins so they are all called the same
name_mapping = {
    "atlantic_ocean": "atlantic_arctic_ocean",
    "atlantic_arctic_ocean": "atlantic_arctic_ocean",
    "atlantic": "atlantic_arctic_ocean",
    "a": "atlantic_arctic_ocean",
    "Atlantic": "atlantic_arctic_ocean"
}

# Function to process data and extract the atlantic_arctic_ocean basin 
def basin_separation_II(data, model):
    atlantic_data = []

    # Extract basin names
    basin_labels = ['atlantic_arctic_ocean', 'atlantic_arctic_extended_ocean', 'indian_pacific_ocean', 'global_ocean']
    basin_indexes = range(len(basin_labels))  # Create a default index range
    
    if model == 'IPSL-CM6A-LR':
        data = data.rename({'3basin': 'basin'})
        basin_labels = ['Global', 'Atlantic', 'Indo-Pacific']
        basin_indexes = range(len(basin_labels))  # Create a default index range    

    elif model == 'NorESM2-LM':
        basin_labels = ['atlantic_arctic_ocean', 'atlantic_arctic_extended_ocean', 'indian_pacific_ocean', 'global_ocean']
        basin_indexes = range(len(basin_labels))  # Create a default index range    
        
    elif 'sector' in data.coords and 'basin' in data.dims:
        basin_indexes = data['basin'].values
        basin_labels = data['sector'].values
        basin_labels = [label.decode('utf-8').strip() for label in basin_labels]
    elif 'basin' in data.dims and 'requested' in data['basin'].attrs:
        basin_indexes = data['basin'].values
        requested_attr = data['basin'].attrs['requested']
        basin_labels = [k.split('=')[0].strip() for k in requested_attr.split(', ')]
    else:
        print(f"{model} : No valid basin names found.")
        return []

    # Create a mapping of indexes to standardized names
    basin_mapping = {}
    
    for idx, label in zip(basin_indexes, basin_labels):
        standardized_name = name_mapping.get(label.lower(), f"unknown_{label}")
        basin_mapping[idx] = standardized_name

    # Extract data for the atlantic_arctic_ocean basin
    for idx, name in basin_mapping.items():

        basin_data = data.isel(basin=idx)  # Extract data for the basin
        if name == "atlantic_arctic_ocean":
            atlantic_data.append(basin_data)

    return atlantic_data

In [33]:
#hfbasins_exp = [] # time average
hfbasins_exp_t = [] # no time average

lat_basin_exp = []

for experiment in experiments:
    hfbasins_model = []
    lat_basin_model = []

    for i, group in enumerate(groups):
        for j, model in enumerate(models_all[i]):
            
            if model in ncc_models:
                model_path = f'{ncc_path}/{group}/{model}'
            elif model in cmip_models:
                model_path = f'{cmip_path}/{group}/{model}'
            else:
                continue

            print(model)

            if model in ['EC-Earth3-Veg','NorCPM1','FGOALS-g3','SAM0-UNICON']: # models I ended up excluding for different reasons each, although they seemed qualified before
                continue
            
            folder_path_0 = f'{model_path}/{experiment}/r1i1p1f1/Omon/hfbasin'
            file_names = sorted(os.listdir(folder_path_0))
            exception_path = sorted(os.listdir(f'{folder_path_0}/{file_names[0]}'))
            
            if 'latest' not in exception_path:
                grid_folder_path = f'{folder_path_0}/{file_names[0]}/{exception_path[0]}'
            else:
                grid_folder_path = f'{folder_path_0}/{file_names[0]}/latest'
            
            grid_file_names = sorted(os.listdir(grid_folder_path))
            
            datasets = [xr.open_dataset(os.path.join(grid_folder_path, file_name), chunks={'time': 10}) 
                        for file_name in grid_file_names if file_name.endswith('.nc')]
            
            if len(datasets) == 1:
                ds = datasets[0]
            else:
                ds = xr.concat(datasets, dim='time')            
            
            #ds_50 = ds.isel(time=slice(start_y, end_y)).mean(dim='time') #time average
            ds_50 = ds.isel(time=slice(start_y, end_y)) # no time average
            
            lat = ds_50['nav_lat'][:, 0].values if model == 'IPSL-CM6A-LR' else ds_50['lat'].values
            dask.compute()
            
            component_values = ds_50['hfbasin']
            #component_values = xr.where(np.isfinite(component_values), component_values, 0) #replacing nans and infs with 0, bc there's no ht happening on land
            atlantic_data = basin_separation_II(component_values, model)
            
            for atlantic_basin_data in atlantic_data:

                if atlantic_basin_data.shape == (332, 1):  #just for the ipsl model- i have to fix how t make it more general later
                    atlantic_basin_data = atlantic_basin_data[:, 0]

                hfbasins_model.append(atlantic_basin_data.values * 10**-15) # has to happen now because interpolation will be difficult with such high values later
                lat_basin_model.append(lat)                
   
    #hfbasins_exp.append(hfbasins_model) # time averaged
    hfbasins_exp_t.append(hfbasins_model) # time-dependent
    
    lat_basin_exp.append(lat_basin_model)

FGOALS-g3
CanESM5
EC-Earth3-Veg
IPSL-CM6A-LR
MPI-ESM1-2-HR
MRI-ESM2-0
NorCPM1
NorESM2-LM
NorESM2-MM
SAM0-UNICON
FGOALS-g3
CanESM5
EC-Earth3-Veg
IPSL-CM6A-LR
MPI-ESM1-2-HR
MRI-ESM2-0
NorCPM1
NorESM2-LM
NorESM2-MM
SAM0-UNICON


In [27]:
# Using time average
hfbasins_exp_interp = []  # Interpolated results storage

for hfbasins_model, lat_basin_model in zip(hfbasins_exp, lat_basin_exp):
    hfbasins_model_interp = []
    
    for hfbasin_data, lat_data in zip(hfbasins_model, lat_basin_model):

        # Convert lat_data to a pandas Series to drop duplicates while keeping indices
        lat_series = pd.Series(lat_data)
        unique_lat_data, unique_indices = np.unique(lat_series, return_index=True)

        # Sort by latitude to ensure proper interpolation
        sorted_indices = np.argsort(unique_lat_data)
        unique_lat_data = unique_lat_data[sorted_indices]
        hfbasin_data = hfbasin_data[unique_indices][sorted_indices]  # Align hfbasin_data

        # Now create DataArray and interpolate
        da = xr.DataArray(hfbasin_data, coords=[unique_lat_data], dims=["lat"])
        da_interp = da.interp(lat=target_lat)
        hfbasins_model_interp.append(da_interp.values)

    hfbasins_exp_interp.append(hfbasins_model_interp)

hfbasins_exp_interp = np.array(hfbasins_exp_interp)

In [32]:
# Save interpolated data
os.chdir('/nird/home/chrikap/6_models')
np.save('hfbasins_exp_atlantic', hfbasins_exp_interp)

In [38]:
# Using time average
hfbasins_exp_t_interp = []  # Interpolated results storage

for hfbasins_model, lat_basin_model in zip(hfbasins_exp_t, lat_basin_exp):
    hfbasins_model_interp = []
    
    for hfbasin_data, lat_data in zip(hfbasins_model, lat_basin_model):
        # Convert lat_data to a pandas Series to drop duplicates while keeping indices
        lat_series = pd.Series(lat_data)
        unique_lat_data, unique_indices = np.unique(lat_series, return_index=True)
        
        # Sort by latitude to ensure proper interpolation
        sorted_indices = np.argsort(unique_lat_data)
        unique_lat_data = unique_lat_data[sorted_indices]
        
        # Convert hfbasin_data to a NumPy array for indexing
        hfbasin_data = np.array(hfbasin_data)
        hfbasins_time_interp = []
        
        for hfbasin_time in hfbasin_data:
            hfbasin_time = np.array(hfbasin_time)[unique_indices][sorted_indices]  # Align hfbasin_data properly
            
            # Flatten if extra dimensions exist
            if hfbasin_time.ndim > 1:
                hfbasin_time = hfbasin_time.squeeze()
            
            # Ensure it aligns with unique_lat_data
            if hfbasin_time.shape[0] != unique_lat_data.shape[0]:
                raise ValueError(f"Mismatch in dimensions: hfbasin_time={hfbasin_time.shape}, unique_lat_data={unique_lat_data.shape}")
            
            # Ensure the correct shape for DataArray
            da = xr.DataArray(hfbasin_time, coords={"lat": unique_lat_data}, dims=["lat"])
            da_interp = da.interp(lat=target_lat)
            
            hfbasins_time_interp.append(da_interp.values)
        
        hfbasins_model_interp.append(hfbasins_time_interp)
    
    hfbasins_exp_t_interp.append(hfbasins_model_interp)

hfbasins_exp_t_interp = np.array(hfbasins_exp_t_interp)

In [41]:
# Save interpolated data
os.chdir('/nird/home/chrikap/6_models')
np.save('hfbasins_exp_atlantic_t', hfbasins_exp_t_interp)